In [1]:
import re
import pandas as pd
from collections import defaultdict


# ==========================================================
# 1. DOCUMENT COLLECTION
# ==========================================================

documents = {
    "D1": """
    Artificial intelligence allows computers to perform tasks that normally
    require human intelligence. Machine learning is an important part of AI.
    """,

    "D2": """
    Machine learning uses algorithms and data to build predictive models.
    Python is widely used for machine learning and data science.
    """,

    "D3": """
    Natural language processing enables computers to understand human language.
    NLP is widely used in chatbots, translation, and information retrieval.
    """,

    "D4": """
    Information retrieval is the process of finding relevant information
    from a collection of documents. Search engines use information retrieval.
    """,

    "D5": """
    Python is a popular programming language for artificial intelligence,
    machine learning, data analysis, and automation.
    """,

    "D6": """
    Deep learning uses neural networks to learn complex patterns from large
    amounts of data. It is used in computer vision and natural language processing.
    """,

    "D7": """
    Database systems store and organize large amounts of information.
    SQL is commonly used to retrieve information from databases.
    """,

    "D8": """
    Search engines use indexing and ranking techniques to retrieve relevant
    documents for user queries. Boolean retrieval uses operators such as AND, OR, and NOT.
    """
}


# ==========================================================
# 2. TOKENIZATION
# ==========================================================

def tokenize(text):

    text = text.lower()

    tokens = re.findall(r'\b[a-z]+\b', text)

    return tokens


# ==========================================================
# 3. TOKENIZE ALL DOCUMENTS
# ==========================================================

tokenized_documents = {}

for doc_id, text in documents.items():

    tokenized_documents[doc_id] = tokenize(text)


# ==========================================================
# 4. BUILD DICTIONARY
# ==========================================================

dictionary = set()

for tokens in tokenized_documents.values():

    dictionary.update(tokens)

dictionary = sorted(dictionary)


# ==========================================================
# 5. BUILD INVERTED INDEX
# ==========================================================

inverted_index = defaultdict(set)

for doc_id, tokens in tokenized_documents.items():

    for term in set(tokens):

        inverted_index[term].add(doc_id)


inverted_index = {
    term: sorted(doc_ids)
    for term, doc_ids in sorted(inverted_index.items())
}


# ==========================================================
# 6. SEARCH SINGLE TERM
# ==========================================================

def search_term(term):

    term = term.lower().strip()

    return set(inverted_index.get(term, []))


# ==========================================================
# 7. BOOLEAN AND
# ==========================================================

def boolean_and(term1, term2):

    return (
        search_term(term1)
        .intersection(search_term(term2))
    )


# ==========================================================
# 8. BOOLEAN OR
# ==========================================================

def boolean_or(term1, term2):

    return (
        search_term(term1)
        .union(search_term(term2))
    )


# ==========================================================
# 9. BOOLEAN NOT
# ==========================================================

def boolean_not(term):

    all_documents = set(documents.keys())

    return (
        all_documents
        .difference(search_term(term))
    )


# ==========================================================
# 10. AND NOT
# ==========================================================

def and_not(term1, term2):

    return (
        search_term(term1)
        .difference(search_term(term2))
    )


# ==========================================================
# 11. GENERAL BOOLEAN QUERY
# ==========================================================

def boolean_query(query):

    query = query.lower().strip()

    if " and " in query:

        terms = query.split(" and ")

        if len(terms) == 2:

            return boolean_and(
                terms[0].strip(),
                terms[1].strip()
            )

    if " or " in query:

        terms = query.split(" or ")

        if len(terms) == 2:

            return boolean_or(
                terms[0].strip(),
                terms[1].strip()
            )

    if query.startswith("not "):

        term = query[4:].strip()

        return boolean_not(term)

    return search_term(query)


# ==========================================================
# 12. DISPLAY DOCUMENTS
# ==========================================================

def display_results(result_documents):

    if not result_documents:

        print("No documents found.")

        return

    print(
        f"\nNumber of matching documents: "
        f"{len(result_documents)}"
    )

    for doc_id in sorted(result_documents):

        print("\n" + "=" * 70)

        print(doc_id)

        print("-" * 70)

        print(documents[doc_id].strip())


# ==========================================================
# 13. DISPLAY SYSTEM INFORMATION
# ==========================================================

print("=" * 70)
print("BASIC INFORMATION RETRIEVAL SYSTEM")
print("=" * 70)

print("\nNumber of documents:", len(documents))

print("Number of unique terms:", len(dictionary))

print("\nSample Dictionary:")
print(dictionary[:20])

print("\nSample Inverted Index:")

for term in list(inverted_index.keys())[:20]:

    print(
        f"{term:20} -> "
        f"{', '.join(inverted_index[term])}"
    )


# ==========================================================
# 14. TEST BOOLEAN RETRIEVAL
# ==========================================================

queries = [
    "machine AND learning",
    "python OR database",
    "information AND retrieval",
    "learning AND NOT python",
    "NOT database"
]

print("\n" + "=" * 70)
print("BOOLEAN RETRIEVAL RESULTS")
print("=" * 70)

for query in queries:

    result = boolean_query(query)

    print("\nQuery :", query)

    print(
        "Documents:",
        sorted(result)
    )

BASIC INFORMATION RETRIEVAL SYSTEM

Number of documents: 8
Number of unique terms: 87

Sample Dictionary:
['a', 'ai', 'algorithms', 'allows', 'amounts', 'an', 'analysis', 'and', 'artificial', 'as', 'automation', 'boolean', 'build', 'chatbots', 'collection', 'commonly', 'complex', 'computer', 'computers', 'data']

Sample Inverted Index:
a                    -> D4, D5
ai                   -> D1
algorithms           -> D2
allows               -> D1
amounts              -> D6, D7
an                   -> D1
analysis             -> D5
and                  -> D2, D3, D5, D6, D7, D8
artificial           -> D1, D5
as                   -> D8
automation           -> D5
boolean              -> D8
build                -> D2
chatbots             -> D3
collection           -> D4
commonly             -> D7
complex              -> D6
computer             -> D6
computers            -> D1, D3
data                 -> D2, D5, D6

BOOLEAN RETRIEVAL RESULTS

Query : machine AND learning
Documents: ['D1', 'D2